# Dew — Calibración `imatrix` en Español y Cuantización Móvil de `empero-ai/Qwen3.8-9B-Distill`

Este notebook ejecuta en una GPU gratuita (Google Colab T4 16 GB o Kaggle 2x T4) el flujo completo de empaquetado de las dos versiones de **Dew**:
1. **`dew-qwen3.8-9b-compact-8gb-iq3_xxs-es.gguf`** (~3,75 GB para móviles de 8 GB de RAM, con tensores de vocabulario español protegidos `q4_k`/`q5_k`).
2. **`dew-qwen3.8-9b-pro-12gb-q4_k_m-es.gguf`** (~5,30 GB para móviles de 12 GB+ de RAM).
3. Publicación directa y gratuita en tu repositorio de Hugging Face (`CeuTryharder/Dew-Qwen3.8-9B-Mobile-ES-GGUF`).

In [ ]:
# Paso 1: Clonar el repositorio de Dew y compilar llama.cpp con aceleración CUDA
!git clone https://github.com/CeuTryharder/dew-asistente-hogar.git
%cd dew-asistente-hogar
!pip install -q huggingface_hub

!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!cmake -B /content/llama.cpp/build -S /content/llama.cpp -DGGML_CUDA=ON -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j $(nproc) --target llama-imatrix llama-quantize llama-cli
import os
os.environ['PATH'] += ':/content/llama.cpp/build/bin'

In [ ]:
# Paso 2: Generar el corpus de calibración en español doméstico (~36.200 tokens) y validar guardarraíles
!python model_pipeline/calibrate_and_package.py

In [ ]:
# Paso 3: Descargar el GGUF maestro de empero-ai/Qwen3.8-9B-Distill y calcular la matriz de importancia (imatrix)
!mkdir -p model_pipeline/base_weights
!huggingface-cli download empero-ai/Qwen3.8-9B-Distill-GGUF --include "*Q8_0.gguf" --local-dir model_pipeline/base_weights

import glob
base_files = glob.glob('model_pipeline/base_weights/*.gguf')
assert base_files, 'No se encontró el archivo GGUF base'
BASE_GGUF = base_files[0]
print('Usando modelo maestro:', BASE_GGUF)

# Calcular dew_qwen38_es.imatrix sobre GPU
!llama-imatrix -m "{BASE_GGUF}" \
  -f model_pipeline/corpus/generated/dew_calibration_es.txt \
  -o model_pipeline/dist/dew_qwen38_es.imatrix \
  -ngl 99 --ctx-size 2048 --chunks 120

In [ ]:
# Paso 4: Generar las dos variantes móviles (8 GB IQ3_XXS y 12 GB+ Q4_K_M) protegiendo tensores de español
!llama-quantize --imatrix model_pipeline/dist/dew_qwen38_es.imatrix \
  --token-embd-type q4_k --output-tensor-type q5_k \
  "{BASE_GGUF}" \
  model_pipeline/dist/dew-qwen3.8-9b-compact-8gb-iq3_xxs-es.gguf IQ3_XXS

!llama-quantize --imatrix model_pipeline/dist/dew_qwen38_es.imatrix \
  --token-embd-type q5_k --output-tensor-type q6_k \
  "{BASE_GGUF}" \
  model_pipeline/dist/dew-qwen3.8-9b-pro-12gb-q4_k_m-es.gguf Q4_K_M

In [ ]:
# Paso 5: Evaluar latencia y cumplimiento de gramáticas GBNF y actualizar el manifiesto con hashes SHA-256
!python model_pipeline/evaluate_quantized_models.py
!python model_pipeline/cloud_quantize_and_publish.py --hf-target-repo CeuTryharder/Dew-Qwen3.8-9B-Mobile-ES-GGUF